In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv("../data/raw/fraud_oracle.csv")

X = df.drop(
    columns=["FraudFound_P", "PolicyNumber", "RepNumber"]
)

y = df["FraudFound_P"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (12336, 30)
Testing: (3084, 30)


In [2]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_train)

print("Baseline classification results:")
print(classification_report(y_train, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_train, y_pred))

Baseline classification results:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     11598
           1       0.00      0.00      0.00       738

    accuracy                           0.94     12336
   macro avg       0.47      0.50      0.48     12336
weighted avg       0.88      0.94      0.91     12336

Confusion matrix:
[[11598     0]
 [  738     0]]


In [3]:
baseline_accuracy = accuracy_score(y_train, y_pred)
baseline_precision = precision_score(y_train, y_pred, zero_division=0)
baseline_recall = recall_score(y_train, y_pred, zero_division=0)
baseline_f1 = f1_score(y_train, y_pred, zero_division=0)

baseline_pr_auc = average_precision_score(
    y_train,
    baseline.predict_proba(X_train)[:, 1]
)

print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"F1-score: {baseline_f1:.4f}")
print(f"Average Precision: {baseline_pr_auc:.4f}")

Accuracy: 0.9402
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000
Average Precision: 0.0598


In [4]:
print("Current features:")
print(X_train.columns.tolist())

print("\nUnique values for selected features:")

for col in ["Year", "PoliceReportFiled", "WitnessPresent", "AgentType"]:
    print(f"\n{col}:")
    print(X_train[col].value_counts())

Current features:
['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year', 'BasePolicy']

Unique values for selected features:

Year:
Year
1994    4949
1995    4152
1996    3235
Name: count, dtype: int64

PoliceReportFiled:
PoliceReportFiled
No     12004
Yes      332
Name: count, dtype: int64

WitnessPresent:
WitnessPresent
No     12266
Yes       70
Name: count, dtype: int64

AgentType:
AgentType
External    12132
Internal      204
Name: count, dtype: int64


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

X_train = X_train.copy()
X_test = X_test.copy()

X_train["Age"] = X_train["Age"].replace(0, np.nan)
X_test["Age"] = X_test["Age"].replace(0, np.nan)

categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

categorical_features += ["Year", "DriverRating"]

numerical_features = [
    col for col in X_train.select_dtypes(include="number").columns
    if col not in categorical_features
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Model preprocessing pipeline ready")

Model preprocessing pipeline ready


In [9]:
decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ))
])

decision_tree.fit(X_train, y_train)

y_train_pred = decision_tree.predict(X_train)

print("Decision Tree — Training Results")
print(classification_report(
    y_train,
    y_train_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, y_train_pred))

Decision Tree — Training Results
              precision    recall  f1-score   support

           0       1.00      0.61      0.76     11598
           1       0.13      0.95      0.23       738

    accuracy                           0.63     12336
   macro avg       0.56      0.78      0.50     12336
weighted avg       0.94      0.63      0.72     12336


Confusion Matrix:
[[7056 4542]
 [  35  703]]


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    decision_tree,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "average_precision": "average_precision"
    }
)

for metric in [
    "precision",
    "recall",
    "f1",
    "average_precision"
]:
    values = scores[f"test_{metric}"]
    print(f"{metric}: {values.mean():.4f} (+/- {values.std():.4f})")

precision: 0.1317 (+/- 0.0040)
recall: 0.9309 (+/- 0.0153)
f1: 0.2308 (+/- 0.0064)
average_precision: 0.1892 (+/- 0.0162)


In [11]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_train)

print("Logistic Regression — Training Results")
print(classification_report(
    y_train,
    logistic_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, logistic_pred))

Logistic Regression — Training Results
              precision    recall  f1-score   support

           0       0.99      0.64      0.78     11598
           1       0.14      0.90      0.24       738

    accuracy                           0.66     12336
   macro avg       0.56      0.77      0.51     12336
weighted avg       0.94      0.66      0.75     12336


Confusion Matrix:
[[7422 4176]
 [  72  666]]


In [12]:
logistic_scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "average_precision": "average_precision"
    }
)

print("Logistic Regression — Cross-Validation")

for metric in ["precision", "recall", "f1", "average_precision"]:
    values = logistic_scores[f"test_{metric}"]
    print(f"{metric}: {values.mean():.4f} (+/- {values.std():.4f})")

Logistic Regression — Cross-Validation
precision: 0.1331 (+/- 0.0053)
recall: 0.8509 (+/- 0.0191)
f1: 0.2302 (+/- 0.0085)
average_precision: 0.1509 (+/- 0.0110)


In [13]:
from sklearn.naive_bayes import GaussianNB
from sklearn.base import clone

nb_preprocessor = clone(preprocessor)
nb_preprocessor.set_params(cat__encoder__sparse_output=False)

naive_bayes_model = Pipeline([
    ("preprocessor", nb_preprocessor),
    ("classifier", GaussianNB())
])

naive_bayes_model.fit(X_train, y_train)

nb_pred = naive_bayes_model.predict(X_train)

print("Naive Bayes — Training Results")
print(classification_report(
    y_train,
    nb_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, nb_pred))

Naive Bayes — Training Results
              precision    recall  f1-score   support

           0       0.99      0.05      0.10     11598
           1       0.06      0.99      0.12       738

    accuracy                           0.11     12336
   macro avg       0.53      0.52      0.11     12336
weighted avg       0.93      0.11      0.10     12336


Confusion Matrix:
[[  598 11000]
 [    7   731]]


In [14]:
nb_scores = cross_validate(
    naive_bayes_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "average_precision": "average_precision"
    }
)

print("Naive Bayes — Cross-Validation")

for metric in ["precision", "recall", "f1", "average_precision"]:
    values = nb_scores[f"test_{metric}"]
    print(f"{metric}: {values.mean():.4f} (+/- {values.std():.4f})")

Naive Bayes — Cross-Validation
precision: 0.0610 (+/- 0.0011)
recall: 0.9648 (+/- 0.0199)
f1: 0.1147 (+/- 0.0021)
average_precision: 0.0662 (+/- 0.0042)
